# CSVデータ作成

##インポート

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

##ウィジェット・変数設定

In [0]:
# ウィジェットの作成
dbutils.widgets.text("P_KIJYUN", "200504", "基準年月 (YYYYMM)") 
p_kijyun = dbutils.widgets.get("P_KIJYUN") 

In [0]:
yyyy = p_kijyun[:4]  
mm = int(p_kijyun[4:6]) 

print(f"集計対象年: {yyyy}")
print(f"集計対象月数: 1月 〜 {mm}月")

##結合

###マスタ結合

In [0]:
# マスタテーブルの読み込み 
mise_mstr = spark.table("training_bs.user_uenishi_sas.mise_mstr").select("MISE_CD", "MISE_NM") 
syohin_name_mstr = spark.table("training_bs.user_uenishi_sas.syohin_name_mstr1").select("SYOHIN_CD1", "SYOHIN_NM1") 

# 全組み合わせを作成 
base_mstr = mise_mstr.crossJoin(syohin_name_mstr) 
display(base_mstr)

###売上データ結合

In [0]:
# 1月〜MM月までのテーブルを結合 
sales_raw = None
for i in range(1, mm + 1):
    ym = f"{yyyy}{str(i).zfill(2)}"
    table_name = f"training_bs.user_uenishi_sas.conv_uri_{ym}"
    
    current_month_df = spark.table(table_name)
    sales_raw = current_month_df if sales_raw is None else sales_raw.union(current_month_df)

display(sales_raw.limit(5))
display(sales_raw.orderBy(F.monotonically_increasing_id(), ascending=False).limit(5))

In [0]:
# 価格マスタとの結合 
kakaku_mstr = spark.table("training_bs.user_uenishi_sas.syohin_kakaku_mstr") 
uriage_detail = sales_raw.join(kakaku_mstr, sales_raw["商品コード"] == kakaku_mstr["syohin_cd"], "inner") 

display(uriage_detail.limit(5))

In [0]:
# 売上額の計算 
uriage_calc = uriage_detail.withColumn("URIAGE_VAL", F.col("売上個数") * F.col("syohin_tanka")) \
                           .withColumn("URI_MM", F.month(F.to_date("売上年月日", "yyyyMMdd"))) \
                           .withColumn("SYOHIN_CD1", F.substring("商品コード", 1, 2)) 

display(uriage_calc.limit(5))

##列展開

In [0]:
# 月ごとに合計値を算出し、列に展開 
monthly_summary = uriage_calc.withColumnRenamed("売上店舗コード", "MISE_CD") \
    .groupBy("SYOHIN_CD1", "MISE_CD") \
    .pivot("URI_MM", list(range(1, 13))) \
    .agg(F.sum("URIAGE_VAL")) 

display(monthly_summary.limit(5))

In [0]:
# 全組み合わせマスタに集計結果をマージ 
final_df = base_mstr.join(monthly_summary, ["SYOHIN_CD1", "MISE_CD"], "left") 

display(final_df.limit(5))

##上下期計算

In [0]:
# 1月〜12月の列に対してループ処理 
for i in range(1, 13):
    col_name = str(i)
    # i <= MM の場合は欠損値を0に、それ以外は保持
    # かつ1000円単位で切り上げ 
    final_df = final_df.withColumn(
        f"URIAGE_{i}",
        F.ceil(F.coalesce(F.col(col_name), F.lit(0)) / 1000) if i <= mm else F.lit(None)
    )

display(final_df.limit(5))

In [0]:
# 上期・下期・年間の算出
final_df = final_df.withColumn("URIAGE_KAMI", sum(F.coalesce(F.col(f"URIAGE_{i}"), F.lit(0)) for i in range(1, 7))) \
                   .withColumn("URIAGE_SHIMO", sum(F.coalesce(F.col(f"URIAGE_{i}"), F.lit(0)) for i in range(7, 13))) \
                   .withColumn("URIAGE_NEN", F.col("URIAGE_KAMI") + F.col("URIAGE_SHIMO"))

display(final_df.limit(5))

In [0]:
# 不要なカラム「1」～「12」を削除し、URIAGE_6の後にURIAGE_SHIMOを配置
# URIAGE_6の後にURIAGE_SHIMOを挿入
uriage_6_idx = cols.index("URIAGE_6")
cols = cols[:uriage_6_idx+1] + ["URIAGE_SHIMO"] + cols[uriage_6_idx+1:]
# 重複しないように
cols = [col for i, col in enumerate(cols) if col != "URIAGE_SHIMO" or i == cols.index("URIAGE_SHIMO")]

final_df = final_df.select(*cols)
display(final_df.limit(5))

##ファイル出力

In [0]:
# 出力先パスの設定
output_path = f"/training_bs/user_uenishi/output/uriage_jisseki_{p_kijyun}"

# 商品カテゴリごとにディレクトリを分けて出力 
final_df.write \
    .partitionBy("SYOHIN_CD1") \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(output_path) 

print(f"出力完了: {output_path}")